# 05 - Evaluación del Modelo con MLflow

## 🎯 Objetivo
Evaluar la calidad del modelo de clustering para determinar el **número óptimo de clústeres (K)**.

Utilizamos dos métodos complementarios:
1. **Método del Codo** (Inercia)
2. **Silhouette Score**

In [ ]:
df = spark.table("gold_customer_features")

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

features = ["Recency", "Frequency", "Monetary"]

assembler = VectorAssembler(
    inputCols=features,
    outputCol="features_raw"
)

df_features = assembler.transform(df)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=True
)

scaler_model = scaler.fit(df_features)
df_scaled = scaler_model.transform(df_features)

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Configurar evaluador de Silhouette
# El Silhouette Score mide qué tan bien está asignado cada punto a su clúster
# Valores cercanos a 1 = clústeres bien separados y compactos
evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="prediction",
    metricName="silhouette"
)

# Rangos de K a evaluar (para método del codo y silhouette)
ks = [2,3,4,5,6,7,8]

results = []

print("🔧 Evaluador configurado para probar K =", list(ks))

## 📉 Parte 5: Método del Codo (Elbow Method)

Utilizamos el **método del codo** para evaluar la inercia del modelo.

### ¿Qué es la Inercia?
Mide qué tan **compactos** son los clústeres. Se define como:
> *Suma de las distancias cuadradas entre cada punto y su centroide más cercano*

### Proceso:
Probamos varios valores de **K (2 a 8)** y graficamos:
- **Eje X:** Número de clústeres (K)
- **Eje Y:** Inercia

### Interpretación:
A medida que aumentamos K, la inercia **disminuye** (más clústeres = puntos más cerca de centroides).

Pero llega un punto donde la **mejora deja de ser significativa**. Ese punto es el **"codo"**.

> 🔑 **El K del codo** representa el equilibrio entre simplicidad y calidad del modelo.

In [ ]:
## 🎯 Parte 6: Silhouette Score

Para **complementar** el análisis del codo, usamos el **Silhouette Score**.

### ¿Qué mide?
Evalúa qué tan **bien está asignado** cada punto a su clúster, considerando:
- **Cohesión:** Qué tan cerca está del centroide de su clúster
- **Separación:** Qué tan lejos está del centroide del clúster más cercano

### Escala de valores:

| Valor | Interpretación |
|-------|--------------|
| **~1** | ✅ Clústeres bien separados y compactos |
| **~0** | ⚠️ Solapamiento entre clústeres |
| **< 0** | ❌ Puntos asignados al clúster incorrecto |

### Uso:
Calculamos el Silhouette Score para cada valor de K. Nos permite validar que el K elegido no solo minimiza la inercia, sino que también genera **clústeres bien definidos**.

## 📊 Parte 7: Tracking con MLflow

Todos los experimentos se registran en **MLflow** para reproducibilidad.

### ¿Qué guardamos para cada K?
- ✅ **Parámetros:** Valor de K
- ✅ **Métricas:** Inercia y Silhouette Score
- ✅ **Modelo:** Artefacto del modelo entrenado

### Beneficios:
- 📋 **Reproducibilidad:** Cualquier experimento puede repetirse
- 📊 **Comparación:** Comparar métricas entre diferentes K
- 🏆 **Mejor modelo:** Identificar fácilmente la configuración óptima

### Experimento: `rfm_clustering_experiment`

In [ ]:
import mlflow

# Configurar experimento en MLflow
mlflow.set_experiment("rfm_clustering_experiment")

# Ejecutar experimentos para cada valor de K
for k in ks:
    with mlflow.start_run(run_name=f"kmeans_k_{k}"):

        # Entrenar modelo K-Means
        kmeans = KMeans(
            k=k,
            seed=42,
            featuresCol="features"
        )

        model = kmeans.fit(df_scaled)
        predictions = model.transform(df_scaled)

        # Calcular métricas
        inertia = model.summary.trainingCost  # Para método del codo
        silhouette = evaluator.evaluate(predictions)  # Calidad de clústeres

        # Registrar en MLflow
        mlflow.log_param("k", k)
        mlflow.log_metric("inertia", inertia)
        mlflow.log_metric("silhouette", silhouette)
        
        # Guardar el modelo entrenado
        mlflow.spark.log_model(model, "kmeans_model")

        results.append((k, inertia, silhouette))
        
        print(f"K={k} → Inertia: {inertia:,.0f}, Silhouette: {silhouette:.3f}")

print(f"\n✅ {len(results)} experimentos registrados en MLflow")

import pandas as pd

# Convertir resultados a DataFrame para visualización
results_df = pd.DataFrame(results, columns=["K", "Inertia", "Silhouette"])
print("📊 Resultados de la evaluación:")
print(results_df.to_string(index=False))

## 📈 Visualización de Resultados

Ahora visualizamos las métricas calculadas para identificar el K óptimo.

In [ ]:
import matplotlib.pyplot as plt

# Gráfica del Método del Codo (Inercia vs K)
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.plot(results_df["K"], results_df["Inertia"], marker='o', linewidth=2, markersize=8)
plt.xlabel("Número de Clusters (K)")
plt.ylabel("Inercia (WSSSE)")
plt.title("📉 Método del Codo")
plt.grid(True, alpha=0.3)

# Gráfica del Silhouette Score
plt.subplot(1, 2, 2)
plt.plot(results_df["K"], results_df["Silhouette"], marker='o', color='green', linewidth=2, markersize=8)
plt.xlabel("Número de Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("🎯 Silhouette Score")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Busca el 'codo' en la gráfica izquierda y el máximo en la derecha")

## ✅ Selección del K Óptimo

Basándonos en ambas métricas:
- **Inercia** (Método del Codo) → buscamos el "codo" donde la mejora se estanca
- **Silhouette Score** → buscamos el **máximo** (mejor separación entre clústeres)

> 📌 **Regla práctica:** El K óptimo suele estar donde ambas métricas concuerdan.

# Determinar K óptimo basado en Silhouette Score
best_k = int(results_df.loc[results_df["Silhouette"].idxmax(), "K"])
best_silhouette = results_df["Silhouette"].max()

print(f"🏆 K ÓPTIMO: {best_k}")
print(f"📊 Silhouette Score: {best_silhouette:.3f}")
print("\n📋 Detalles del mejor modelo:")
print("-" * 40)
best_row = results_df.loc[results_df["Silhouette"].idxmax()]
print(best_row.to_string())

print(f"\n✅ Usa K={best_k} en el notebook 04_clustering_model.ipynb")

# Guardar K óptimo para uso en notebook 04
import json
k_config = {"optimal_k": best_k, "silhouette_score": best_silhouette}
with open("/tmp/kmeans_config.json", "w") as f:
    json.dump(k_config, f)

print(f"\n💾 Configuración guardada: K={best_k}")

In [ ]:
## ✅ Conclusión del Notebook 05

### Evaluación Completada:
1. ✅ Preparación de datos (VectorAssembler + StandardScaler)
2. ✅ Evaluación con Método del Codo (K=2..8)
3. ✅ Evaluación con Silhouette Score
4. ✅ Tracking en MLflow (`rfm_clustering_experiment`)
5. ✅ Visualización de métricas
6. ✅ Selección de K óptimo

### Resultado:
> El valor de K seleccionado debe usarse en el notebook `04_clustering_model.ipynb` para el modelo final.

### Todos los experimentos están registrados en MLflow:
- Accede a la UI de MLflow en Databricks para comparar métricas
- Cada run contiene: K, Inercia, Silhouette Score y el modelo